# Justificativa com Llama

Os candidatos do notebook 05 vão para um modelo local no Ollama, então nenhum dado sai da máquina.

O modelo avalia um candidato por vez e recebe apenas a ementa e o enunciado. A pergunta parte do precedente: a ementa aplica, contraria ou não trata desta regra? Isso funciona melhor em ementas com vários assuntos, em que perguntar se "a questão é a mesma" fazia o modelo errar a Súmula 676.

Toda resposta traz uma justificativa, mesmo quando não há consonância. Ela é marcada como jurídica quando a resposta depende de comparar regras ou matérias, e como fática quando depende de um fato concreto do caso.

Depois, o código confere a resposta fora do modelo. O trecho citado precisa existir na ementa, e uma justificativa vazia ou um número que não esteja nos textos fornecidos gera um alerta.

## Qual modelo usar

O padrão é o `llama2:7b`, o Llama de 7B pedido no projeto. Na avaliação do notebook 07 ele responde quase sempre a mesma coisa, então o exemplo também roda com o `llama3.1:8b`. Para trocar de modelo, basta mudar `MODELO_LLM`.

In [1]:
EXEMPLOS_BUSCA = False
%run ./05_busca_consonancia.ipynb

In [2]:
import json
import time

MODELO_LLM = "llama2:7b"
MODELO_COMPARACAO = "llama3.1:8b"

SISTEMA = """Você é assistente jurídico e responde em português do Brasil. O PRECEDENTE enuncia uma regra jurídica. Verifique se a EMENTA trata dessa mesma regra, usando somente esses dois textos.

Preencha, nesta ordem:
- regra_do_precedente: a regra do precedente, em uma frase curta.
- trecho_da_ementa: copie da ementa o trecho que trata dessa regra. Se nenhum trecho tratar dela, deixe vazio.
- tipo_justificativa:
  - JURIDICA: a resposta depende de comparar regras, teses, normas ou matérias do direito. Use também quando os textos tratam de matérias jurídicas diferentes (por exemplo, usucapião e divórcio) ou quando concluem em sentidos iguais ou opostos sobre a mesma regra.
  - FATICA: a regra do precedente seria a mesma da ementa, e a resposta depende de um fato concreto do caso (por exemplo, a idade de alguém, o lugar ou o que a parte fez), que faz o caso se enquadrar ou não na regra.
- justificativa: obrigatória em todos os casos, inclusive em NAO_TRATA. Em até duas frases, explique o porquê da sua resposta, indicando o que a ementa diz e o que o precedente diz.
- relacao:
  - APLICA: a ementa decide no mesmo sentido da regra.
  - CONTRARIA: a ementa trata da mesma regra e decide no sentido oposto.
  - NAO_TRATA: a ementa não trata dessa regra. Palavras em comum ou mesma área do direito não bastam."""

ESQUEMA = {
    "type": "object",
    "properties": {
        "regra_do_precedente": {"type": "string"},
        "trecho_da_ementa": {"type": "string"},
        "tipo_justificativa": {"type": "string", "enum": ["JURIDICA", "FATICA"]},
        "justificativa": {"type": "string", "minLength": 20},
        "relacao": {"type": "string", "enum": ["APLICA", "CONTRARIA", "NAO_TRATA"]},
    },
    "required": ["regra_do_precedente", "trecho_da_ementa", "tipo_justificativa", "justificativa", "relacao"],
}
ROTULOS = {"APLICA": "CONSONANTE", "CONTRARIA": "DIVERGENTE", "NAO_TRATA": "SEM CONSONÂNCIA"}


def normalizar(texto: str) -> str:
    return re.sub(r"\s+", " ", texto).strip().lower()


def numeros_sem_lastro(gerado: str, fontes: str) -> list[str]:
    """Números de 2+ dígitos (súmula, tema, lei, artigo) usados na resposta e ausentes dos textos fornecidos."""
    padrao = r"\d[\d./]*\d"
    return sorted(set(re.findall(padrao, gerado)) - set(re.findall(padrao, fontes)))


def justificar(ementa: str, citacao: str, enunciado: str, modelo: str = MODELO_LLM) -> dict:
    inicio = time.perf_counter()
    resposta = ollama.chat(
        model=modelo,
        messages=[
            {"role": "system", "content": SISTEMA},
            {"role": "user", "content": f"EMENTA:\n{ementa}\n\nPRECEDENTE ({citacao}):\n{enunciado}"},
        ],
        format=ESQUEMA,
        options={"temperature": 0, "seed": 42, "num_ctx": 4096, "num_predict": 800},
    )
    alertas = []
    try:
        dados = json.loads(resposta.message.content)
    except json.JSONDecodeError:
        dados = {}
        alertas.append("resposta fora do formato")
    relacao = ROTULOS.get(dados.get("relacao"), "INDEFINIDA")
    trecho = str(dados.get("trecho_da_ementa", "")).strip()
    justificativa = str(dados.get("justificativa", "")).strip()
    if not justificativa:
        alertas.append("sem justificativa")
    if relacao in ("CONSONANTE", "DIVERGENTE") and normalizar(trecho).strip(" .…") not in normalizar(ementa):
        alertas.append("o trecho citado não existe na ementa")
    if relacao in ("CONSONANTE", "DIVERGENTE") and not trecho:
        alertas.append("nenhum trecho citado")
    sem_lastro = numeros_sem_lastro(f"{trecho} {justificativa}", f"{ementa} {citacao} {enunciado}")
    if sem_lastro:
        alertas.append(f"números sem lastro nos textos: {sem_lastro}")
    return {
        "citacao": citacao,
        "relacao": relacao,
        "trecho": trecho,
        "tipo_justificativa": dados.get("tipo_justificativa", "INDEFINIDA"),
        "justificativa": justificativa,
        "alertas": alertas,
        "segundos": round(time.perf_counter() - inicio, 1),
    }


def analisar(query: str, modelo: str = MODELO_LLM, sumulas: int = 3, precedentes: int = 2, exibir_busca: bool = True) -> list[dict]:
    resultado = consonancia(query)
    if exibir_busca:
        mostrar(resultado)
    ementa = limpar_query(query)
    candidatos = resultado["Súmulas"][:sumulas] + resultado["Precedentes com tese"][:precedentes]
    pareceres = [justificar(ementa, c.citacao, c.enunciado, modelo) for c in candidatos]
    linhas = [
        f"### Avaliação do modelo `{modelo}`\n",
        "_Gerada por modelo local e sem valor de decisão: confira cada enunciado na fonte oficial._\n",
    ]
    for p in pareceres:
        aviso = f" **Alerta: {'; '.join(p['alertas'])}.**" if p["alertas"] else ""
        linhas.append(f"- **{p['citacao']}** → {p['relacao']} ({p['segundos']} s). Justificativa {p['tipo_justificativa'].lower()}: {p['justificativa']}{aviso}")
    display(Markdown("\n".join(linhas)))
    return pareceres

## Exemplo com a ementa sobre prisão preventiva

In [3]:
if globals().get("EXEMPLOS_JUSTIFICATIVA", True):
    pareceres = analisar(QUERY_EXEMPLO)

### Súmulas

- **[Súmula 676/STJ](https://scon.stj.jus.br/SCON/pesquisar.jsp?b=SUMU&sumula=676)**: aderência forte (0.698), vigente. Em razão da Lei n. 13.964/2019, não é mais possível ao juiz, de ofício, decretar ou converter prisão em flagrante em prisão preventiva.
- **[Súmula 409/STJ](https://scon.stj.jus.br/SCON/pesquisar.jsp?b=SUMU&sumula=409)**: aderência moderada (0.604), vigente. Em execução fiscal, a prescrição ocorrida antes da propositura da ação pode ser decretada de ofício (art. 219, § 5º, do CPC).
- **[Súmula 668/STJ](https://scon.stj.jus.br/SCON/pesquisar.jsp?b=SUMU&sumula=668)**: aderência moderada (0.532), vigente. Não é hediondo o delito de porte ou posse de arma de fogo de uso permitido, ainda que com numeração, marca ou qualquer outro sinal de identificação raspado, suprimido ou adulterado.
- **[Súmula 267/STJ](https://scon.stj.jus.br/SCON/pesquisar.jsp?b=SUMU&sumula=267)**: aderência moderada (0.603), vigente. A interposição de recurso, sem efeito suspensivo, contra decisão condenatória não obsta a expedição de mandado de prisão.
- **[Súmula 630/STJ](https://scon.stj.jus.br/SCON/pesquisar.jsp?b=SUMU&sumula=630)**: aderência moderada (0.547), alterada. A incidência da atenuante da confissão espontânea no crime de tráfico ilícito de entorpecentes quando o acusado admitir a posse ou a propriedade para uso próprio, negando a prática do tráfico de drogas, deve ocorrer em proporção inferior à que seria devida no caso de confissão plena.

### Precedentes com tese

- **[Tema 1098/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1098&cod_tema_final=1098)**: aderência moderada (0.606), Trânsito em Julgado. 1 - O Acordo de Não Persecução Penal constitui um negócio jurídico processual penal instituído por norma que possui natureza processual, no que diz respeito à possibilidade de composição entre as partes com o fim de evitar a instauração da ação penal, e, de outro lado, natureza material em razão da previsão de extinção da punibilidade de quem cumpre os deveres estabelecidos no acordo (art. 28-A, §
- **[Tema 134/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=134&cod_tema_final=134)**: aderência moderada (0.649), Trânsito em Julgado. Em execução fiscal, a prescrição ocorrida antes da propositura da ação pode ser decretada de ofício (art. 219, § 5º, do CPC).
- **[Tema 1259/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1259&cod_tema_final=1259)**: aderência moderada (0.552), Trânsito em Julgado. A majorante do art. 40, inciso IV, da Lei n. 11.343/2006 aplica-se quando há nexo finalístico entre o uso da arma e o tráfico de drogas, sendo a arma usada para garantir o sucesso da atividade criminosa, hipótese em que o crime de porte ou posse ilegal de arma é absorvido pelo tráfico. Do contrário, o delito previsto no Estatuto do Desarmamento é considerado crime autônomo, em concurso material co
- **[Tema 1347/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1347&cod_tema_final=1347)**: aderência moderada (0.618), Trânsito em Julgado. A regressão cautelar de regime prisional é medida de caráter provisório e está autorizada pelo poder geral de cautela do juízo da execução, podendo ser aplicada, mediantefundamentação idônea, até a apuração definitiva da falta.
- **[Tema 1167/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1167&cod_tema_final=1167)**: aderência moderada (0.610), Trânsito em Julgado. "A audiência prevista no art. 16 da Lei 11.340/2006 tem por objetivo confirmar a retratação, não a representação, e não pode ser designada de ofício pelo juiz. Sua realização somente é necessária caso haja manifestação do desejo da vítima de se retratar trazida aos autos antes do recebimento da denúncia".

### Em andamento (sem tese firmada)

- **[Tema 1431/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1431&cod_tema_final=1431)**: aderência forte (0.666), Em Julgamento. Definir se a solicitação, sem a efetiva entrega do entorpecente ao destinatário no estabelecimento prisional, caracteriza ato preparatório, impunível em razão da atipicidade da conduta, ou se configura conduta típica de tráfico de drogas pela aplicação do art. 29 do Código Penal.
- **[Tema 1407/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1407&cod_tema_final=1407)**: aderência moderada (0.578), Afetado. Definir, em relação à causa de aumento de pena disposta no art. 157, §2º-A, I, do Código Penal, se: 1) é necessária apreensão de arma de fogo; 2) é necessária a perícia da arma de fogo; 3) é necessária tanto a apreensão quanto a perícia; 4) se, na ausência de apreensão e perícia, outros meios probatórios podem ser considerados hábeis para comprovar o uso do artefato.
- **[Tema 1163/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=1163&cod_tema_final=1163)**: aderência moderada (0.595), Afetado. Saber se a simples fuga do réu para dentro da residência ao avistar os agentes estatais e/ou a mera existência de denúncia anônima acerca da possível prática de delito no interior do domicílio,desacompanhada de outros elementos preliminares indicativos de crime, constituem ou não, por si sós, fundadas razões (justa causa) a autorizar o ingresso dos policiais em seu domicílio, sem prévia autorizaçã

### Inativos (não usar como fundamento)

- **[Tema 583/STJ](https://processo.stj.jus.br/repetitivos/temas_repetitivos/pesquisa.jsp?novaConsulta=true&tipo_pesquisa=T&cod_tema_inicial=583&cod_tema_final=583)**: aderência forte (0.698), Cancelado. Questiona-se a possibilidade de concessão de liberdade provisória a preso, em flagrante, pela suposta prática do crime de tráfico ilícito de drogas (art. 33, caput,da Lei n.º 11.343/2006), haja visto a vedação expressa do art. 44 da Lei n.º 11.343/2006.
- **Controvérsia 438/STJ**: aderência forte (0.653), Vinculada a Tema. Estabelecer se, a despeito da guarda municipal não desempenhar a função de policiamento ostensivo, ela pode prender quem esteja em flagrante delito, respaldada no art. 301 do Código de Processo Penal.


### Avaliação do modelo `llama2:7b`

_Gerada por modelo local e sem valor de decisão: confira cada enunciado na fonte oficial._

- **Súmula 676/STJ** → CONSONANTE (19.9 s). Justificativa juridica: A decisão do precedente é baseada na interpretação da Lei n. 13.964/2019, que limita a possibilidade de decretar ou converter prisão em flagrante em prisão preventiva de ofício. A ementa, por sua vez, estabelece que a decisão agravada deve ser mantida, pois a conversão da prisão em flagrante em preventiva não é diferente da decretação da prisão preventiva como uma primeira prisão. Portanto, a regra do precedente é consistente com a interpretação da Lei n. 13.964/2019.
- **Súmula 409/STJ** → CONSONANTE (15.8 s). Justificativa juridica: A decisão do precedente é baseada na interpretação da Lei n. 13.964/2019, que estabelece que a decretação da prisão preventiva de ofício é possível apenas após o ocorrência de audiência de custódia. Neste caso, a prisão em flagrante não constitui uma audiência de custódia, pois não houve ocorrência de uma audiência formal para a determinação da prisão. Portanto, a decisão do precedente é que a decretação da prisão preventiva de ofício não é possível em este caso.
- **Súmula 668/STJ** → CONSONANTE (13.7 s). Justificativa juridica: A ementa decide que a conversão da prisão em flagrante em preventiva não é diferente da decretação da prisão preventiva como uma primeira prisão, o que é importante para entender a regra do precedente. A Lei n. 13.964/2019 não distingue entre a prisão em flagrante e a decretação da prisão preventiva, e isso é reforçado pela decisão do STJ em Súmula 668/STJ, que estabelece que o porte ou posse de arma de fogo de uso permitido não é hediondo, independentemente da forma como está raspado ou suprimido.
- **Tema 1098/STJ** → CONSONANTE (13.7 s). Justificativa juridica: O princípio da retroatividade da norma pena benéfica estabelece que as normas processuais que entraram em vigor após a comissão do crime, podem ser aplicadas a esses casos, mesmo que o réu não tenha confessado ainda. Isso é válido desde que o pedido tenha sido feito antes do trânsito em julgado da condenação. **Alerta: o trecho citado não existe na ementa.**
- **Tema 134/STJ** → CONSONANTE (16.1 s). Justificativa juridica: A decisão do precedente é baseada na interpretação da Lei n. 13.964/2019, que estabelece que a decretação da prisão preventiva de ofício é possível apenas após o ocorrência de audiência de custódia. Neste caso, a prisão em flagrante não constitui uma audiência de custódia, pois não houve ocorrência de audiência de custódia antes da decretação da prisão preventiva. Portanto, a decisão do precedente é que a decretação da prisão preventiva de ofício não é possível em este caso.

In [4]:
if globals().get("EXEMPLOS_JUSTIFICATIVA", True):
    pareceres_comparacao = analisar(QUERY_EXEMPLO, modelo=MODELO_COMPARACAO, exibir_busca=False)

### Avaliação do modelo `llama3.1:8b`

_Gerada por modelo local e sem valor de decisão: confira cada enunciado na fonte oficial._

- **Súmula 676/STJ** → CONSONANTE (12.0 s). Justificativa juridica: A ementa trata da impossibilidade de decretação de prisão preventiva de ofício, o que é confirmado pelo precedente que também menciona a Lei n. 13.964/2019 como base para essa regra.
- **Súmula 409/STJ** → SEM CONSONÂNCIA (5.8 s). Justificativa fatica: A ementa trata de prisão preventiva e conversão de prisão em flagrante, enquanto o precedente trata de prescrição em execução fiscal. Além disso, a ementa não menciona a possibilidade de decretação de prescrição de ofício em execução fiscal.
- **Súmula 668/STJ** → SEM CONSONÂNCIA (6.5 s). Justificativa juridica: A ementa não trata da regra do precedente, pois trata de prisão preventiva e porte ilegal de arma de fogo de uso permitido, enquanto o precedente trata especificamente da hediondez do delito de porte ou posse de arma de fogo de uso permitido.
- **Tema 1098/STJ** → SEM CONSONÂNCIA (8.2 s). Justificativa fatica: A ementa não trata da regra do precedente, pois discute a prisão preventiva e a conversão de prisão em flagrante em preventiva, enquanto o precedente trata do Acordo de Não Persecução Penal (ANPP) e sua aplicação em processos penais em andamento.
- **Tema 1259/STJ** → SEM CONSONÂNCIA (8.6 s). Justificativa juridica: A ementa não trata da regra do precedente, pois trata de prisão preventiva, porte ilegal de arma, tráfico de drogas e associação para o tráfico, enquanto o precedente trata da aplicação da majorante do art. 40, inciso IV, da Lei n. 11.343/2006.